# Air Traffic Control Conflict Detection: A Trajectory-Based Approach

This document presents an implementation of conflict detection for Air Traffic Control (ATC) systems, demonstrating the trajectory-based methodology for identifying separation minima violations. The system detects when two aircraft come within 5 nautical miles horizontally and 1,000 feet vertically of each other simultaneously.

## Conceptual Foundation

In air traffic management, a loss of separation (conflict) is defined as a situation wherein two aircraft violate established separation minima. The standard separation criteria employed in this implementation are:

- Lateral separation: 5 nautical miles (horizontal distance)
- Vertical separation: 1,000 feet (altitude difference)

It is essential to note that both conditions must be satisfied concurrently for a conflict to be identified. An aircraft pair maintaining adequate lateral separation but insufficient vertical separation (or vice versa) does not constitute a conflict under this definition.

## Input Data Requirements

The conflict detection system requires three distinct data components:

### 1. Airspace Graph (GraphML Format)

The airspace structure is represented as a directed graph containing waypoints (nodes) and connections (edges).

### 2. Aircraft Data (JSON Format)

Aircraft specifications including unique identifiers and performance characteristics.

### 3. Flight Plans (JSON Format)

Intended routes for each aircraft, specifying the sequence of waypoints to be traversed.

## Function 1: Airspace Graph Loading

In [1]:
import networkx as nx

def load_airspace_graph(graphml_path: str) -> nx.Graph:
    """
    Loads the airspace structure from a GraphML file.
    
    Args:
        graphml_path: Path to the GraphML file
        
    Returns:
        A NetworkX graph with nodes (waypoints) and edges (connections)
        
    Requirements:
        - Nodes must have 'latitude' and 'longitude' attributes
        - Edges must have 'distance_nm' attribute
    """
    graph = nx.read_graphml(graphml_path)
    
    # Validate required attributes
    for node_id, node_data in graph.nodes(data=True):
        if 'latitude' not in node_data or 'longitude' not in node_data:
            raise ValueError(f"Node {node_id} missing latitude/longitude attributes")
    
    for source, target, edge_data in graph.edges(data=True):
        if 'distance_nm' not in edge_data:
            raise ValueError(f"Edge {source}->{target} missing distance_nm attribute")
    
    return graph

## Data Classes

This Python code defines the core data models for the conflict detection system. Using `dataclass` structures, it establishes typed representations of flight parameters, trajectories, and separation monitoring:

* **`Aircraft` & `FlightPlan**`: Define the static and scheduled properties of a flight, including airspeed (knots), ordered route waypoints, departure timing, and cruise altitude.
* **`TrajectoryPoint`**: Captures a discrete 4D point in space-time ($x, y$ in nautical miles, altitude in feet, and time in seconds) to model aircraft movement along its route.
* **`Conflict`**: Encapsulates a detected loss-of-separation event between two aircraft, recording the predicted time, horizontal/vertical separation distances, and affected route segments.

In [2]:
from typing import List, Dict, Any
from dataclasses import dataclass

@dataclass
class Aircraft:
    id: str
    speed_kt: float

@dataclass  
class FlightPlan:
    aircraft_id: str
    route: List[str]
    departure_time_s: float
    cruise_altitude_ft: float

@dataclass
class Conflict:
    aircraft_1: str
    aircraft_2: str
    time_s: float
    horizontal_distance_nm: float
    vertical_distance_ft: float
    segment_1: List[str]
    segment_2: List[str]

@dataclass
class TrajectoryPoint:
    node_id: str
    time_s: float
    x_nm: float
    y_nm: float
    altitude_ft: float

## Function 2: Conflict Detection Algorithm

The following implementation provides the primary conflict detection functionality. This algorithm systematically examines all pairs of aircraft trajectories to identify violations of separation minima.

In [3]:
import math
from itertools import combinations


def build_trajectory(plan: FlightPlan, aircraft: Aircraft, graph: nx.Graph) -> List[TrajectoryPoint]:
    """
    Builds a time-based trajectory for an aircraft following its flight plan.
    
    Assumptions:
    - Constant speed between waypoints
    - Constant altitude (cruise_altitude_ft)
    - Straight-line segments between waypoints
    - Simple equirectangular projection for coordinates
    """
    speed_kt = aircraft.speed_kt
    altitude_ft = plan.cruise_altitude_ft
    
    # Calculate projection origin (average of all nodes)
    latitudes = [float(node["latitude"]) for node in graph.nodes.values()]
    longitudes = [float(node["longitude"]) for node in graph.nodes.values()]
    lat0 = sum(latitudes) / len(latitudes)
    lon0 = sum(longitudes) / len(longitudes)
    
    trajectory = []
    current_time = plan.departure_time_s
    
    for i, node_id in enumerate(plan.route):
        node = graph.nodes[node_id]
        latitude = float(node["latitude"])
        longitude = float(node["longitude"])
        
        # Project to NM coordinates
        x_nm = (longitude - lon0) * 60.0 * math.cos(math.radians(lat0))
        y_nm = (latitude - lat0) * 60.0
        
        trajectory.append(TrajectoryPoint(
            node_id=node_id,
            time_s=current_time,
            x_nm=x_nm,
            y_nm=y_nm,
            altitude_ft=altitude_ft
        ))
        
        # Calculate time to next node
        if i < len(plan.route) - 1:
            next_node_id = plan.route[i + 1]
            edge = graph[node_id][next_node_id]
            distance_nm = float(edge["distance_nm"])
            current_time += distance_nm / speed_kt * 3600.0  # Convert to seconds
    
    return trajectory

def position_at(p0: TrajectoryPoint, p1: TrajectoryPoint, time_s: float) -> tuple:
    """Linear interpolation between two trajectory points."""
    if abs(p1.time_s - p0.time_s) < 1e-9:
        ratio = 0.0
    else:
        ratio = (time_s - p0.time_s) / (p1.time_s - p0.time_s)
    
    x = p0.x_nm + ratio * (p1.x_nm - p0.x_nm)
    y = p0.y_nm + ratio * (p1.y_nm - p0.y_nm)
    altitude = p0.altitude_ft + ratio * (p1.altitude_ft - p0.altitude_ft)
    
    return x, y, altitude

def closest_approach(
    a0: TrajectoryPoint, a1: TrajectoryPoint,
    b0: TrajectoryPoint, b1: TrajectoryPoint,
    t0: float, t1: float
) -> tuple:
    """
    Finds the closest approach between two trajectory segments during time overlap.
    Returns: (time_s, horizontal_distance_nm, vertical_distance_ft)
    """
    candidates = [t0, t1]
    
    # Positions at start and end of overlap
    ax0, ay0, aa0 = position_at(a0, a1, t0)
    bx0, by0, ba0 = position_at(b0, b1, t0)
    ax1, ay1, aa1 = position_at(a0, a1, t1)
    bx1, by1, ba1 = position_at(b0, b1, t1)
    
    # Relative position vectors
    rx0, ry0 = ax0 - bx0, ay0 - by0
    rx1, ry1 = ax1 - bx1, ay1 - by1
    
    # Relative velocity vector
    vx, vy = rx1 - rx0, ry1 - ry0
    
    # Find closest approach parameter
    denominator = vx * vx + vy * vy
    
    if denominator > 1e-12:
        u = -((rx0 * vx) + (ry0 * vy)) / denominator
        if 0.0 <= u <= 1.0:
            candidates.append(t0 + u * (t1 - t0))
    
    # Find best (closest) time
    best_time = candidates[0]
    best_horizontal = float("inf")
    best_vertical = float("inf")
    
    for t in candidates:
        ax, ay, aa = position_at(a0, a1, t)
        bx, by, ba = position_at(b0, b1, t)
        
        horizontal = math.hypot(ax - bx, ay - by)
        vertical = abs(aa - ba)
        
        if horizontal < best_horizontal:
            best_time = t
            best_horizontal = horizontal
            best_vertical = vertical
    
    return best_time, best_horizontal, best_vertical

def detect_conflicts(
    aircraft_list: List[Aircraft],
    flight_plans: List[FlightPlan],
    graph: nx.Graph,
    lateral_threshold_nm: float = 5.0,
    vertical_threshold_ft: float = 1000.0
) -> List[Conflict]:
    """
    Detects all conflicts between aircraft trajectories.
    
    A conflict exists when two aircraft come within:
    - lateral_threshold_nm (default: 5.0 NM) horizontally
    - AND vertical_threshold_ft (default: 1000 ft) vertically
    
    Args:
        aircraft_list: List of Aircraft objects
        flight_plans: List of FlightPlan objects
        graph: NetworkX graph from load_airspace_graph()
        lateral_threshold_nm: Horizontal separation threshold in NM
        vertical_threshold_ft: Vertical separation threshold in feet
        
    Returns:
        List of Conflict objects for all detected conflicts
    """
    # Build aircraft lookup
    aircraft_by_id = {a.id: a for a in aircraft_list}
    
    # Validate and build trajectories
    trajectories = {}
    validation_errors = []
    
    for plan in flight_plans:
        # Validate flight plan
        errors = []
        
        if plan.aircraft_id not in aircraft_by_id:
            errors.append(f"unknown aircraft_id: {plan.aircraft_id}")
        
        if len(plan.route) < 2:
            errors.append("route must contain at least two nodes")
        else:
            for node_id in plan.route:
                if node_id not in graph:
                    errors.append(f"unknown route node: {node_id}")
            
            for source, target in zip(plan.route[:-1], plan.route[1:]):
                if not graph.has_edge(source, target):
                    errors.append(f"missing graph edge: {source} -> {target}")
        
        if errors:
            validation_errors.append({"aircraft_id": plan.aircraft_id, "errors": errors})
        else:
            trajectories[plan.aircraft_id] = build_trajectory(
                plan, aircraft_by_id[plan.aircraft_id], graph
            )
    
    # Detect conflicts
    conflicts = []
    
    for ac1, ac2 in combinations(sorted(trajectories.keys()), 2):
        traj1 = trajectories[ac1]
        traj2 = trajectories[ac2]
        
        # Check all pairs of trajectory segments
        for p1_start, p1_end in zip(traj1[:-1], traj1[1:]):
            for p2_start, p2_end in zip(traj2[:-1], traj2[1:]):
                # Find time overlap
                overlap_start = max(p1_start.time_s, p2_start.time_s)
                overlap_end = min(p1_end.time_s, p2_end.time_s)
                
                if overlap_start > overlap_end:
                    continue  # No overlap, no conflict possible
                
                # Calculate closest approach during overlap
                time_s, horizontal_nm, vertical_ft = closest_approach(
                    p1_start, p1_end, p2_start, p2_end, overlap_start, overlap_end
                )
                
                # Check if this violates separation minima
                if (horizontal_nm <= lateral_threshold_nm and 
                    vertical_ft <= vertical_threshold_ft):
                    conflicts.append(Conflict(
                        aircraft_1=ac1,
                        aircraft_2=ac2,
                        time_s=round(time_s, 3),
                        horizontal_distance_nm=round(horizontal_nm, 3),
                        vertical_distance_ft=round(vertical_ft, 1),
                        segment_1=[p1_start.node_id, p1_end.node_id],
                        segment_2=[p2_start.node_id, p2_end.node_id]
                    ))
    
    return conflicts

## Test Data Files

Sample data files are provided to demonstrate the conflict detection system. These test cases are designed to illustrate the core functionality with minimal complexity.

Required files are:

- **Sample Airspace Graph**: `sample_airspace.graphml`
- **Sample Aircraft Specifications**: `sample_aircraft.json`
- **Sample Flight Plans**: `sample_flight_plans.json`

## Execution Example

The following code demonstrates how to execute the conflict detection system with the provided test data:

In [8]:
import json
from dataclasses import asdict

def test_conflict_detection(airspace_graph: str, aircrafts: str, flight_plan:str):
    # Load test data
    graph = load_airspace_graph(airspace_graph)
    
    with open(aircrafts) as f:
        aircraft_data = json.load(f)
    aircraft_list = [Aircraft(**ac) for ac in aircraft_data]
    
    with open(flight_plan) as f:
        flight_plan_data = json.load(f)
    flight_plans = [FlightPlan(**fp) for fp in flight_plan_data]
    
    # Detect conflicts
    conflicts = detect_conflicts(aircraft_list, flight_plans, graph)
    
    # Print results
    print(f"Detected {len(conflicts)} conflicts:")
    for conflict in conflicts:
        print(f"  - {conflict.aircraft_1} vs {conflict.aircraft_2}")
        print(f"    Time: {conflict.time_s}s")
        print(f"    Horizontal: {conflict.horizontal_distance_nm} NM")
        print(f"    Vertical: {conflict.vertical_distance_ft} ft")
        print(f"    Segments: {conflict.segment_1} vs {conflict.segment_2}\n")

### No conflict example

Two aircraft at different altitudes, demonstrating that vertical separation alone prevents conflict detection.

In [9]:
test_conflict_detection("sample_airspace.graphml", "sample_aircraft.json", "sample_no_conflict_flight_plans.json")

Detected 0 conflicts:


### One conflict example

Two aircraft flying directly toward each other on the same route at the same altitude, producing a conflict.

In [10]:
test_conflict_detection("sample_airspace.graphml", "sample_aircraft.json", "sample_conflict_flight_plans.json")

Detected 1 conflicts:
  - AC1 vs AC2
    Time: 112.5s
    Horizontal: 0.0 NM
    Vertical: 0.0 ft
    Segments: ['A', 'B'] vs ['B', 'A']



### Two conflict example

Both aircraft oscillate between A and B at the same speed and altitude, crossing paths twice at the same location but different times, resulting in two distinct conflict detections.

In [12]:
test_conflict_detection("sample_airspace.graphml", "sample_aircraft.json", "sample_conflict_2_flight_plans.json")

Detected 2 conflicts:
  - AC1 vs AC2
    Time: 112.5s
    Horizontal: 0.0 NM
    Vertical: 0.0 ft
    Segments: ['A', 'B'] vs ['B', 'A']

  - AC1 vs AC2
    Time: 337.5s
    Horizontal: 0.0 NM
    Vertical: 0.0 ft
    Segments: ['B', 'A'] vs ['A', 'B']

